# 02 · Compute LCSE for a ligand with AIMNet2-CPCM

Requires `models/b973c_cpcm_ens_f.jpt` (see `models/README.md`). A GPU is recommended: on CPU, a 300-conformer search
for a 50-atom ligand takes about 15 minutes.

Protocol (as in the paper):
1. **Bound-state energy**: relax bond lengths and angles of the pose with every rotatable-bond dihedral frozen at its deposited value.
2. **Global minimum**: generate conformers (here RDKit ETKDG; the paper used up to 5000 Omega conformers), optimize each with AIMNet2-CPCM, keep the lowest.
3. `LCSE = E_bound − E_global`, both in CPCM water at the input protonation state.

In [ ]:
from rdkit import Chem
from lcse import AIMNet2CPCM, ligand_strain, load_conformers, EV_TO_KCAL
import torch; print('cuda available:', torch.cuda.is_available())

Pick a ligand: the first entry of the released set, or your own SDF with explicit hydrogens and the intended protonation state.

In [ ]:
mol = next(load_conformers('bound'))          # or: Chem.MolFromMolFile('pose.sdf', removeHs=False)
name = mol.GetProp('_Name')
print(name, mol.GetNumAtoms(), 'atoms, net charge', sum(a.GetFormalCharge() for a in mol.GetAtoms()))
Chem.MolFromSmiles(Chem.MolToSmiles(Chem.RemoveHs(mol)))

In [ ]:
calc = AIMNet2CPCM(member=0)                   # one ensemble member, as in the production search
res = ligand_strain(mol, calc, n_conformers=300)
print(f"E_bound  = {res['E_bound_eV']:.4f} eV\nE_global = {res['E_global_eV']:.4f} eV   ({res['n_conformers']} conformers optimized)")
print(f"LCSE = {res['lcse_kcal']:.2f} kcal/mol")

Compare with the deposited value (the released table uses the 4-member ensemble mean and a 5000-conformer Omega search, so small differences are expected; a lower global minimum here would lower the strain).

In [ ]:
from lcse import load_master_table
df = load_master_table()
print('deposited LCSE:', round(df.loc[name, 'lcse_kcal'], 2), 'kcal/mol')
res['bound'].write(f'{name}_bound_relaxed.xyz'); res['global'].write(f'{name}_global_min.xyz')

## Ensemble uncertainty

Use the full ensemble (`member=None`) to get the spread across the four networks on each endpoint; the paired difference cancels the shared model bias.

In [ ]:
ens = AIMNet2CPCM(member=None); ens.set_charge(sum(a.GetFormalCharge() for a in mol.GetAtoms()))
for tag, at in [('bound', res['bound']), ('global', res['global'])]:
    at = at.copy(); at.calc = ens; e = at.get_potential_energy()
    print(f"{tag:6s} E = {e:.4f} eV  ensemble std = {ens.results['energy_std'] * EV_TO_KCAL:.2f} kcal/mol")